# Evaluating Laya on `LocalLLaMA/typed-decisions` Benchmark (Single T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NandhaKishorM/laya/blob/main/notebooks/laya_eval_typed_decisions_colab.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Model-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-LocalLLaMA%2Ftyped--decisions-green)](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)

This notebook evaluates **Laya** (`convaiinnovations/laya`) head-to-head against **TypeSafe Jev** on the independent [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark.

### What is `typed-decisions`?
It is the standard public benchmark for System One decision models, covering 4 production workflows:
1. `agent_trace_observability`: Assess agent executions for human intervention and urgency.
2. `customer_service`: Route assistant responses and actions from multi-turn customer dialogues.
3. `invoice_processing`: Review vendor invoices against purchase orders and delivery receipts.
4. `security_incidents`: Determine containment, investigation, or dismissal of security alerts.

Each test case contains **5 typed questions** over a shared state. The entire benchmark consists of **400 cases (2,000 typed decisions)**.


## 1. GPU Check & Setup
Ensure a T4 GPU is selected in `Runtime -> Change runtime type -> T4 GPU`.


In [ ]:
!nvidia-smi
import os, torch
assert torch.cuda.is_available(), "No GPU detected! Set Runtime -> Change runtime type -> T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"Connected to: {props.name} | Total VRAM: {props.total_memory / 1e9:.1f} GB")


## 2. Install Dependencies
Install `laya`, `transformers`, `datasets`, and evaluation libraries.


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy
import laya, torch, datasets, pandas as pd
print("Laya version:", laya.__version__)
print("PyTorch version:", torch.__version__)


## 3. Load Laya Model
Load the fine-tuned Laya model from Hugging Face on CUDA.


In [ ]:
MODEL_ID = "convaiinnovations/laya"
print(f"Loading Laya model from {MODEL_ID}...")

agent = laya.load(MODEL_ID, device="cuda")
print("Model successfully loaded on GPU!")


## 4. Load the `LocalLLaMA/typed-decisions` Benchmark
We load all 400 test cases across the four workflows.


In [ ]:
from datasets import load_dataset
import json

print("Downloading LocalLLaMA/typed-decisions benchmark (split: test)...")
ds = load_dataset("LocalLLaMA/typed-decisions", "all", split="test")

print(f"Loaded {len(ds)} benchmark cases.")
print("Sample workflow breakdown:")
df_summary = pd.DataFrame(ds)
print(df_summary["workflow"].value_counts())


## 5. Run Laya Evaluation Loop
Evaluate all 400 test cases (2,000 decisions) and measure per-case latency.


In [ ]:
import time, math
import numpy as np

predictions = []
latencies_ms = []

print("Starting evaluation of 400 benchmark cases...")
t_start = time.time()

for i, row in enumerate(ds):
    case_id = row["id"]
    workflow = row["workflow"]
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    
    t0 = time.perf_counter()
    res = agent.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    
    predictions.append({
        "id": case_id,
        "workflow": workflow,
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })
    
    if (i + 1) % 50 == 0 or (i + 1) == len(ds):
        print(f"  Progress: {i + 1}/{len(ds)} cases evaluated | Average latency: {np.mean(latencies_ms):.1f} ms/case")

total_time = time.time() - t_start
print(f"\nEvaluation complete in {total_time:.1f}s across {len(ds) * 5} decisions!")
print(f"P50 Latency: {np.percentile(latencies_ms, 50):.1f} ms/case | P95 Latency: {np.percentile(latencies_ms, 95):.1f} ms/case")


## 6. Compute Official Benchmark Metrics
We calculate the exact metrics defined by `LocalLLaMA/typed-decisions`: Accuracy, Soft Accuracy, Brier Score, ECE, KL Divergence, Total Variation (TV), and Score MAE.


In [ ]:
from laya.common import ece_score

accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]
        
        # 1. Choice / Multi-class
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = g_ans["label"]
            
            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(p_ans["confidence"])
            
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            
            # Soft accuracy: sum(p_i * g_i)
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            # Brier score: sum((p - g)^2)
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            # Total Variation: 0.5 * sum(|p - g|)
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            # KL divergence: sum(g * log(g / p))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))
            
        # 2. Noul / Boolean
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = g_ans["label"]
            
            pred_label = "true" if p_val >= 0.5 else "false"
            is_corr = float(pred_label == str(gold_label).lower())
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(max(p_val, 1.0 - p_val))
            
            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])
            
            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))
            
        # 3. Score / Ordinal Rubric
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            
            # Discrete level accuracy
            p_lvl = int(round(p_score))
            g_lvl = int(round(g_score))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(p_ans.get("confidence", 0.5))

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_mae = np.mean(score_maes) if score_maes else 0.0
laya_latency = np.percentile(latencies_ms, 50)

print("--- Laya Benchmark Results on typed-decisions ---")
print(f"Accuracy      : {laya_acc:.3f}")
print(f"Soft Accuracy : {laya_soft_acc:.3f}")
print(f"Brier Score   : {laya_brier:.3f}")
print(f"ECE           : {laya_ece:.3f}")
print(f"Score MAE     : {laya_mae:.3f}")
print(f"KL Divergence : {laya_kl:.3f}")
print(f"Latency (p50) : {laya_latency:.1f} ms/case")


## 7. Head-to-Head Comparison with TypeSafe Jev
Compare Laya directly against the baseline numbers published on `LocalLLaMA/typed-decisions`.


In [ ]:
comparison_data = [
    {
        "Model": "TypeSafe Jev 1.13.0",
        "Kind": "general",
        "Accuracy": 0.727,
        "Soft Acc": 0.580,
        "Brier": 0.148,
        "ECE": 0.144,
        "Score MAE": 0.391,
        "ms/case": 710,
        "Cost/Case": "$0.0004 (API)"
    },
    {
        "Model": "Laya (Ours)",
        "Kind": "general",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "Score MAE": round(laya_mae, 3),
        "ms/case": round(laya_latency, 1),
        "Cost/Case": "$0.00 (Self-Hosted)"
    },
    {
        "Model": "ModernBERT-base (149M)",
        "Kind": "specialist",
        "Accuracy": 0.646,
        "Soft Acc": 0.542,
        "Brier": 0.119,
        "ECE": 0.179,
        "Score MAE": 0.444,
        "ms/case": 349,
        "Cost/Case": "$0.00"
    },
    {
        "Model": "Teacher Self-Agreement",
        "Kind": "ceiling",
        "Accuracy": 0.735,
        "Soft Acc": "-",
        "Brier": "-",
        "ECE": "-",
        "Score MAE": "-",
        "ms/case": "-",
        "Cost/Case": "-"
    }
]

df_comp = pd.DataFrame(comparison_data)
print("=== HEAD-TO-HEAD BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))


## 8. Export Benchmark Report
Save evaluation metrics as JSON and Markdown for reference.


In [ ]:
report = {
    "benchmark": "LocalLLaMA/typed-decisions",
    "model": MODEL_ID,
    "n_cases": len(ds),
    "n_decisions": len(ds) * 5,
    "metrics": {
        "accuracy": float(laya_acc),
        "soft_accuracy": float(laya_soft_acc),
        "brier_score": float(laya_brier),
        "ece": float(laya_ece),
        "score_mae": float(laya_mae),
        "kl_divergence": float(laya_kl),
        "total_variation": float(laya_tv),
        "latency_p50_ms": float(laya_latency),
        "latency_p95_ms": float(np.percentile(latencies_ms, 95))
    }
}

with open("typed_decisions_eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved report to typed_decisions_eval_report.json!")
